# AdaBoost (Adaptive Boosting)

**AdaBoost** is an ensemble learning method that combines many **weak learners** (models only slightly better than random guessing) into one **strong learner**.

### How it works (step by step):
1. Train a weak learner (e.g. a **decision stump** — a tree with `max_depth=1`) on the data.
2. Check which samples it **got wrong**.
3. **Increase the weight** of the misclassified samples so the next learner pays more attention to them.
4. Train the next weak learner on the re-weighted data.
5. Repeat for `n_estimators` rounds.
6. Combine all weak learners using a **weighted vote** — learners that performed better get a louder voice.

> **Analogy:** Imagine a team of students taking a test. After the first student answers, a teacher highlights the questions they got wrong. The next student focuses harder on those. After 50 students, you combine their answers — weighting the best students more — and get a very accurate result.

---

### Key Parameters

| Parameter | What it controls |
|---|---|
| `n_estimators` | **How many** weak learners to chain together sequentially |
| `learning_rate` | **How much** each weak learner contributes to the final prediction |

#### `n_estimators=50`
The number of weak learners (stumps) to train one after another.
- Each new stump **focuses on the mistakes** of all the previous ones.
- `50` means: "chain together 50 decision stumps, each correcting the last."
- **Too few** → underfitting (not enough correction rounds)
- **Too many** → risk of overfitting (especially on noisy data)

#### `learning_rate=1.0`
A shrinkage factor that scales the contribution (vote weight) of each weak learner:

$$\text{effective\_weight} = \text{learning\_rate} \times \text{learner's computed weight}$$

- `learning_rate=1.0` → full contribution (no shrinkage)
- `learning_rate=0.1` → each learner contributes only **10%** of its vote

**Trade-off:** A lower learning rate needs **more estimators** to reach the same accuracy, but often **generalizes better** (less overfitting). Think of it as a "speed dial" — lower = slower but smoother learning.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Loading and Splitting the Dataset
data = load_iris()
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Defining the Weak Learner
base_classifier = DecisionTreeClassifier(max_depth=1)

# Creating and Training the AdaBoost Classifier
adaboost_classifier = AdaBoostClassifier(
                                base_classifier, 
                                n_estimators=50, 
                                learning_rate=1.0, 
                                random_state=42
                            )
adaboost_classifier.fit(X_train, y_train)

# Making Predictions and Calculating Accuracy
y_pred = adaboost_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9333333333333333


### Mini Example: Effect of `n_estimators` and `learning_rate`

Let's compare different combinations to see the trade-off in action:

In [2]:
configs = [
    {"n_estimators": 1,   "learning_rate": 1.0},   # just 1 stump – very weak
    {"n_estimators": 10,  "learning_rate": 1.0},   # 10 stumps – getting better
    {"n_estimators": 50,  "learning_rate": 1.0},   # 50 stumps – solid
    {"n_estimators": 50,  "learning_rate": 0.1},   # 50 stumps but slow learning
    {"n_estimators": 200, "learning_rate": 0.1},   # compensate low lr with more estimators
]

print(f"{'n_estimators':>14} | {'learning_rate':>14} | {'Accuracy':>8}")
print("-" * 46)

for cfg in configs:
    clf = AdaBoostClassifier(
        DecisionTreeClassifier(max_depth=1),
        n_estimators=cfg["n_estimators"],
        learning_rate=cfg["learning_rate"],
        random_state=42
    )
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{cfg['n_estimators']:>14} | {cfg['learning_rate']:>14} | {acc:>8.4f}")

  n_estimators |  learning_rate | Accuracy
----------------------------------------------
             1 |            1.0 |   0.6333
            10 |            1.0 |   1.0000
            50 |            1.0 |   0.9333
            50 |            0.1 |   1.0000
           200 |            0.1 |   1.0000
